<a href="https://colab.research.google.com/github/varakalicheva-hub/compling/blob/main/%D0%94%D0%B8%D0%BF%D0%BB%D0%BE%D0%BC_%D1%81%D0%B1%D0%BE%D1%80_%D0%BA%D0%BE%D1%80%D0%BF%D1%83%D1%81%D0%B0_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time
import csv
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementClickInterceptedException

# ===== НАСТРОЙКИ =====
START_URL = "https://otzovik.com/reviews/skyeng_ru-shkola_izucheniya_inostrannogo_yazika_cherez_internet/"
CSV_FILE = "otzovik_skyeng_reviews.csv"
MAX_PAGES = 179                # всего страниц
DELAY = 2                      # задержка между действиями (сек)
PAGE_LOAD_DELAY = 5            # задержка после загрузки страницы
# ======================

options = webdriver.ChromeOptions()
options.add_argument("--disable-blink-features=automation")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
driver = webdriver.Chrome(options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

def load_existing_reviews(filename):
    existing_urls = set()
    if os.path.exists(filename):
        with open(filename, 'r', encoding='utf-8-sig') as f:
            reader = csv.DictReader(f)
            for row in reader:
                if 'URL' in row and row['URL']:
                    existing_urls.add(row['URL'].strip())
    return existing_urls

def save_review_to_csv(username, review_text, review_date, review_url, filename):
    # Заменяем переносы строк на пробелы и схлопываем множественные пробелы
    review_text = ' '.join(review_text.splitlines())
    review_text = ' '.join(review_text.split())

    file_exists = os.path.exists(filename)
    with open(filename, 'a', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f, delimiter=',', quotechar='"',
                            quoting=csv.QUOTE_ALL, lineterminator='\n')
        if not file_exists:
            writer.writerow(['Имя пользователя', 'Отзыв', 'Дата', 'URL'])
        writer.writerow([username, review_text, review_date, review_url])

def scrape_page(driver, existing_urls):
    collected = 0
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.item"))
        )
    except TimeoutException:
        print("Не найдены отзывы на странице.")
        return 0

    review_blocks = driver.find_elements(By.CSS_SELECTOR, "div.item")
    print(f"Найдено отзывов на странице: {len(review_blocks)}")

    main_window = driver.current_window_handle

    for idx, block in enumerate(review_blocks, 1):
        # --- Имя пользователя ---
        try:
            username_elem = block.find_element(By.CSS_SELECTOR, "a.user-login span")
            username = username_elem.text.strip()
        except NoSuchElementException:
            username = "Неизвестно"

        # --- Дата публикации ---
        try:
            date_elem = block.find_element(By.CSS_SELECTOR, "div.review-postdate span")
            review_date = date_elem.text.strip()
        except NoSuchElementException:
            review_date = "Неизвестно"

        # --- Ссылка на полный отзыв ---
        try:
            read_link = block.find_element(By.CSS_SELECTOR, "a.review-btn.review-read-link")
            review_url = read_link.get_attribute("href")
        except NoSuchElementException:
            print(f"   Пропускаем блок #{idx}: нет ссылки на отзыв")
            continue

        if review_url in existing_urls:
            print(f"   Отзыв {review_url} уже есть, пропускаем")
            continue

        # --- Открываем отзыв в новой вкладке ---
        driver.execute_script("window.open(arguments[0]);", review_url)
        time.sleep(DELAY)
        driver.switch_to.window(driver.window_handles[-1])
        time.sleep(PAGE_LOAD_DELAY)

        # --- Полный текст отзыва ---
        full_text = ""
        try:
            text_elem = driver.find_element(By.CSS_SELECTOR, "div.review-body")
            full_text = text_elem.text.strip()
        except NoSuchElementException:
            try:
                text_elem = driver.find_element(By.CSS_SELECTOR, "div.review-full-text")
                full_text = text_elem.text.strip()
            except NoSuchElementException:
                print(f"   Не удалось найти текст отзыва на странице {review_url}")

        if full_text:
            save_review_to_csv(username, full_text, review_date, review_url, CSV_FILE)
            existing_urls.add(review_url)
            collected += 1
            print(f"   Собран отзыв #{idx}: {username} ({review_date})")
        else:
            print(f"   Не удалось получить текст для {review_url}")

        # --- Закрываем вкладку и возвращаемся к списку ---
        driver.close()
        driver.switch_to.window(main_window)
        time.sleep(DELAY)

    return collected

def go_to_next_page(driver):
    try:
        # Ищем ссылку, внутри которой есть span с текстом "Далее"
        next_btn = driver.find_element(By.XPATH, "//a[.//span[contains(text(),'Далее')]]")
        driver.execute_script("arguments[0].scrollIntoView();", next_btn)
        time.sleep(1)
        next_btn.click()
        return True
    except NoSuchElementException:
        # Если не нашли по тексту, ищем по классам pager-next или next
        try:
            next_btn = driver.find_element(By.CSS_SELECTOR, "a.pager-next, a.next")
            driver.execute_script("arguments[0].scrollIntoView();", next_btn)
            time.sleep(1)
            next_btn.click()
            return True
        except NoSuchElementException:
            print("Кнопка 'Далее' не найдена. Вероятно, это последняя страница.")
            return False
    except ElementClickInterceptedException:
        driver.execute_script("arguments[0].click();", next_btn)
        return True

def main():
    existing_urls = load_existing_reviews(CSV_FILE)
    print(f"Загружено {len(existing_urls)} уже собранных отзывов.")

    driver.get(START_URL)
    time.sleep(PAGE_LOAD_DELAY)

    # Проверка на капчу
    if "captcha" in driver.current_url or "Вы робот?" in driver.title:
        print("Обнаружена капча. Пожалуйста, вручную введите код и нажмите 'Я не робот'.")
        print("Скрипт ожидает, пока капча не будет пройдена...")

        while True:
            if "captcha" not in driver.current_url and "Вы робот?" not in driver.title:
                try:
                    driver.find_element(By.CSS_SELECTOR, "form.captcha-form")
                    time.sleep(1)
                    continue
                except NoSuchElementException:
                    break
            time.sleep(2)

        print("Капча пройдена. Ожидаем загрузки отзывов...")
        time.sleep(3)

        try:
            WebDriverWait(driver, 30).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "div.item"))
            )
            print("Отзывы загружены, продолжаем сбор.")
        except TimeoutException:
            print("Не удалось дождаться загрузки отзывов после капчи. Пробуем перейти по START_URL.")
            driver.get(START_URL)
            time.sleep(PAGE_LOAD_DELAY)
            try:
                WebDriverWait(driver, 30).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "div.item"))
                )
                print("Отзывы загружены после перехода.")
            except TimeoutException:
                print("Отзывы не найдены. Возможно, сайт изменил структуру. Выход.")
                driver.quit()
                return

    page_num = 1
    total_collected = 0

    while True:
        print(f"\n--- Обрабатывается страница {page_num} ---")
        collected = scrape_page(driver, existing_urls)
        total_collected += collected
        print(f"Собрано на странице: {collected}, всего: {total_collected}")

        if page_num >= MAX_PAGES:
            print(f"Достигнут лимит страниц ({MAX_PAGES}). Завершаем.")
            break

        if not go_to_next_page(driver):
            break

        page_num += 1
        time.sleep(PAGE_LOAD_DELAY)

    print(f"\nСбор завершён. Всего новых отзывов: {total_collected}. Всего в файле теперь: {len(existing_urls)}")
    driver.quit()

if __name__ == "__main__":
    main()